# Module 17: Experimentation & Model Training

**Snowpark ML Fundamentals — Day 1 Afternoon Session 1**

---

## Learning Objectives

You already know how to train models and tune hyperparameters (Modules 03, 15).
Today's focus: **How do you organize experiments so your team can reproduce,
compare, and learn from each one?**

| Concept | What You'll Learn |
|---------|-------------------|
| Feature set experiments | Same algorithm, different inputs → different outcomes |
| Provenance metadata | Register experiments with comments, version metadata, tags, and metrics |
| Systematic comparison | Use the Model Registry to filter, sort, and select |

## Estimated Time: 45 minutes

---

## 17.1 Setup & Experiment Framing (5 min)

We'll run **3 experiments** — same algorithm (XGBoost), but different feature sets.
The Model Registry becomes our experiment tracker.

In [ ]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../../../src')

import logging
import warnings

import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

logger = logging.getLogger("module_17")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

from snowflake.snowpark import functions as F

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data, explore_dataframe
from snowpark_fundamentals.preprocessing.transformers import (
    apply_preprocessing_pipeline, build_preprocessing_pipeline,
)
from snowpark_fundamentals.preprocessing.feature_engineering import (
    create_derived_features, create_interaction_features,
)
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import (
    evaluate_binary_classifier, get_feature_importance,
)
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model,
    compare_model_versions, set_model_tags, set_model_comment, set_model_version_metadata,
    get_model_metrics, list_versions,
)

session = create_session(env_path='../../../.env')
logger.info("Session created.")

In [ ]:
# --- Schema isolation ---
STUDENT_NAME = "YOUR_NAME"  # <-- CHANGE THIS to your name (e.g., "MARIA")

session.sql("USE ROLE MLDS_ROLE_D").collect()
session.sql("USE DATABASE MLDS_D").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {STUDENT_NAME}_ONSITE_TRAINING").collect()
session.sql(f"USE SCHEMA {STUDENT_NAME}_ONSITE_TRAINING").collect()
logger.info(f"Working in schema: {STUDENT_NAME}_ONSITE_TRAINING")

# Generate dataset
churn_df = create_customer_churn_dataset(session, n_rows=5000)
train_df, test_df = split_data(churn_df, train_ratio=0.8)
logger.info(f"Train: {train_df.count()} rows, Test: {test_df.count()} rows")

In [ ]:
# --- Feature columns ---
NUMERIC_COLS = [
    'AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'SUPPORT_TICKETS',
    'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS', 'TOTAL_CHARGES',
]
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
LABEL_COL = 'CHURNED'

# Registry setup
current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

MODEL_NAME = f"{STUDENT_NAME}_EXPERIMENT_TRACKER"

# Cleanup from previous runs
try:
    delete_model(registry, MODEL_NAME)
except Exception:
    pass

logger.info(f"Model name: {MODEL_NAME}")

---

## 17.2 Experiment A: Baseline — Numeric Features Only (8 min)

**Concept:** Start simple. Train with only numeric features as our baseline.
Every subsequent experiment must beat this to justify its added complexity.

In [ ]:
# --- Experiment A: Numeric features only (baseline) ---
model_a = train_model(
    train_df, NUMERIC_COLS, LABEL_COL,
    model_type='xgboost',
    model_params={'n_estimators': 100, 'max_depth': 6},
)
preds_a = predict(model_a, test_df)
metrics_a = evaluate_binary_classifier(preds_a, LABEL_COL, 'PREDICTION')

logger.info("Experiment A — Numeric Only:")
for k, v in metrics_a.items():
    logger.info(f"  {k}: {v:.4f}")

In [ ]:
# --- Register with provenance metadata ---
log_model(
    registry=registry,
    model=model_a.to_xgboost(),
    model_name=MODEL_NAME,
    version_name='EXP_A_NUMERIC_ONLY',
    sample_input=test_df.select(NUMERIC_COLS).limit(10),
    metrics=metrics_a,
)

set_model_comment(
    registry, MODEL_NAME,
    "Experiment A: Numeric features only (7 features). Baseline approach — "
    "no preprocessing, no feature engineering. XGBoost n_estimators=100, max_depth=6.",
    version_name='EXP_A_NUMERIC_ONLY',
)

set_model_version_metadata(
    registry, MODEL_NAME, 'EXP_A_NUMERIC_ONLY',
    {
        'provenance': {
            'experiment_type': 'feature_selection',
            'feature_set': 'numeric_only',
            'feature_count': 7,
            'dataset_rows': 5000,
            'algorithm': 'xgboost',
            'preprocessing': 'none',
        }
    },
)

# Model tags should only describe labels shared by every version in this tracker.
set_model_tags(registry, MODEL_NAME, {
    'experiment_family': 'feature_selection',
    'dataset_name': 'synthetic_customer_churn',
    'algorithm_family': 'xgboost',
    'module': '17_experimentation',
})

logger.info(f"Registered {MODEL_NAME} / EXP_A_NUMERIC_ONLY with provenance.")
logger.info("Notice: version name describes the EXPERIMENT, not just 'V1'.")
logger.info("Version metadata captures WHAT changed. Comments capture WHY.")
logger.info("Model tags are reserved for tracker-level labels and discovery.")

---

## 17.3 Experiment B: Add Categorical Features (10 min)

**Concept:** Encoding categorical columns (contract type, payment method, plan type)
gives the model access to discrete segments. Does this extra information improve
predictions?

Your turn — preprocess the data, train, evaluate, and register.

In [ ]:
# TODO 1: Build a preprocessing pipeline that includes both numeric AND categorical columns
# This encodes categoricals (OrdinalEncoder) and scales numerics (StandardScaler)
# Hint: Fit on train with build_preprocessing_pipeline(...), then transform test with
#   apply_preprocessing_pipeline(..., transformers)

processed_train, transformers = ____
processed_test = apply_preprocessing_pipeline(test_df, NUMERIC_COLS, CATEGORICAL_COLS, transformers)
processed_feature_cols = [f"{c}_SCALED" for c in NUMERIC_COLS] + [f"{c}_ENCODED" for c in CATEGORICAL_COLS]

logger.info(f"Processed feature columns: {processed_feature_cols}")
logger.info(f"Feature count: {len(processed_feature_cols)}")

In [ ]:
# TODO 2: Train XGBoost on the processed data and evaluate
# Use only the encoded/scaled feature columns from TODO 1
# Hint: Same train_model() call as Experiment A, but use processed_feature_cols

model_b = ____

preds_b = predict(model_b, processed_test)
metrics_b = evaluate_binary_classifier(preds_b, LABEL_COL, 'PREDICTION')

logger.info("Experiment B — Numeric + Categorical:")
for k, v in metrics_b.items():
    logger.info(f"  {k}: {v:.4f}")

In [ ]:
# TODO 3: Register Experiment B with provenance metadata
# Follow the same pattern as Experiment A:
#   1. log_model() with version_name='EXP_B_WITH_CATEGORICALS'
#   2. set_model_comment() describing what changed from Experiment A
#   3. set_model_version_metadata() with provenance['feature_set']='numeric_and_categorical'
# Note: model-level tags were already set in Experiment A
# Hint: For log_model, use model_b.to_xgboost(), metrics=metrics_b, and
#   sample_input=processed_test.select(processed_feature_cols).limit(10)

____  # log_model

____  # set_model_comment — describe what's different from Experiment A

____  # set_model_version_metadata — store experiment_type, feature_set, feature_count

logger.info(f"Registered {MODEL_NAME} / EXP_B_WITH_CATEGORICALS")

In [ ]:
# --- Validation B ---
assert metrics_b is not None, "TODO 2: model_b must be trained"
assert isinstance(metrics_b, dict), "TODO 2: metrics_b should be a dict"
assert set(metrics_b.keys()) == {'accuracy', 'precision', 'recall', 'f1_score'}, \
    "TODO 2: metrics_b should have 4 keys"
logger.info("Experiment B: All validations passed!")

---

## 17.4 Experiment C: Add Derived & Interaction Features (10 min)

**Concept:** Feature engineering creates new signals from existing columns:
- **Derived features:** tenure buckets, high-value flags, risk flags
- **Interaction features:** products of column pairs (e.g., tenure × charges)

Do engineered features beat raw features?

In [ ]:
# TODO 4: Create derived features and interaction features on training data
# Hint: See notebook 02
#   - create_derived_features(df) adds: CHARGES_PER_MONTH_TENURE, TENURE_BUCKET,
#     HIGH_VALUE_FLAG, AT_RISK_FLAG, TICKETS_PER_TENURE
#   - create_interaction_features(df, [('TENURE_MONTHS', 'MONTHLY_CHARGES')])
#     adds: TENURE_MONTHS_X_MONTHLY_CHARGES
# Apply BOTH to train_df and test_df

enriched_train = ____
enriched_train = ____

enriched_test = create_derived_features(test_df)
enriched_test = create_interaction_features(enriched_test, [('TENURE_MONTHS', 'MONTHLY_CHARGES')])

logger.info(f"Enriched columns ({len(enriched_train.columns)}):")
logger.info(enriched_train.columns)

In [ ]:
# TODO 5: Train on enriched features, evaluate, and register as 'EXP_C_ENGINEERED'
# Use the original NUMERIC_COLS plus the new derived columns as features
# Hint: The new columns are CHARGES_PER_MONTH_TENURE, HIGH_VALUE_FLAG,
#   AT_RISK_FLAG, TENURE_MONTHS_X_MONTHLY_CHARGES
# Note: Skip TENURE_BUCKET (categorical string) to keep this simple

engineered_cols = NUMERIC_COLS + [
    'CHARGES_PER_MONTH_TENURE', 'HIGH_VALUE_FLAG',
    'AT_RISK_FLAG', 'TENURE_MONTHS_X_MONTHLY_CHARGES',
]

model_c = ____

preds_c = predict(model_c, enriched_test)
metrics_c = evaluate_binary_classifier(preds_c, LABEL_COL, 'PREDICTION')

logger.info("Experiment C — Engineered Features:")
for k, v in metrics_c.items():
    logger.info(f"  {k}: {v:.4f}")

# Register
____  # log_model with version 'EXP_C_ENGINEERED', metrics=metrics_c, model_c.to_xgboost()

____  # set_model_comment — describe the feature engineering applied

____  # set_model_version_metadata — store provenance for the engineered feature experiment

logger.info(f"Registered {MODEL_NAME} / EXP_C_ENGINEERED")

In [ ]:
# --- Validation C ---
assert metrics_c is not None, "TODO 5: model_c must be trained"
assert isinstance(metrics_c, dict), "TODO 5: metrics_c should be a dict"
versions_df = list_versions(registry, MODEL_NAME)
assert len(versions_df) == 3, "All 3 experiments should be registered"
logger.info("Experiment C: All validations passed!")

---

## 17.5 Systematic Comparison & Takeaways (7 min)

Now let's compare all 3 experiments side-by-side using the Model Registry.

In [ ]:
# TODO 6: Compare all 3 experiment versions side-by-side
# Hint: See notebook 10 — use compare_model_versions(registry, MODEL_NAME, version_list)

comparison = ____

# Format as table
rows = []
for item in comparison:
    row = {'experiment': item['version']}
    row.update(item['metrics'])
    rows.append(row)

results_df = pd.DataFrame(rows).set_index('experiment')
logger.info(results_df.to_string())
logger.info(f"\nBest experiment by F1: {results_df['f1_score'].idxmax()}")

In [ ]:
# --- Experiment metadata in the registry ---
versions_df = list_versions(registry, MODEL_NAME)
logger.info("Registered experiments:")
columns_to_show = [c for c in ['name', 'comment', 'metadata'] if c in versions_df.columns]
logger.info(versions_df[columns_to_show].to_string())

---

## Key Takeaways

1. **Model Registry as experiment tracker** — Each version represents a distinct experiment,
   not just a hyperparameter variant. Version names describe WHAT changed.

2. **Structured provenance for reproducibility** — Every experiment stores a comment plus
   version metadata describing the feature set, preprocessing, and rationale.

3. **Model tags for discovery** — Keep tags for tracker-level labels shared by every version.
   Use version names, comments, and metadata to distinguish experiments within that tracker.

4. **Systematic comparison** — `compare_model_versions()` gives you a side-by-side view.
   The best F1 score tells you which feature set to take forward.

**Next session:** Now we need to **validate** the winning experiment before promoting it
to production. → Module 18: Model Validation & Registration

In [ ]:
# --- Cleanup ---
delete_model(registry, MODEL_NAME)
logger.info(f"Cleaned up {MODEL_NAME}")

In [ ]:
session.close()